In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:

import os

ROOT = "/kaggle/input"

for root, dirs, files in os.walk(ROOT):
    level = root.replace(ROOT, "").count(os.sep)
    if level > 3:
        dirs[:] = []
        continue

    print("  " * level + os.path.basename(root) + "/")

    for f in files[:15]:
        print("  " * (level + 1) + f)


In [ ]:

import glob

print("CSV:", len(glob.glob(
    "/kaggle/input/**/*.csv", recursive=True
)))

print("Arrow:", len(glob.glob(
    "/kaggle/input/**/*.arrow", recursive=True
)))

print("DatasetDict:", glob.glob(
    "/kaggle/input/**/dataset_dict.json",
    recursive=True
))

print("ZIP:", glob.glob(
    "/kaggle/input/**/*.zip",
    recursive=True
))


In [ ]:

from datasets import load_from_disk
import os

INPUT_DIR = (
    "/kaggle/input/datasets/embehacker/"
    "vsfc-experiment-backup"
)

BASE_DIR = "/kaggle/working/vsfc_llm"
os.makedirs(BASE_DIR, exist_ok=True)

# Đọc dữ liệu gốc
ds = load_from_disk(
    os.path.join(INPUT_DIR, "data")
)

# Đọc dữ liệu đã tách từ bằng VnCoreNLP
segmented = load_from_disk(
    os.path.join(INPUT_DIR, "segmented_data")
)

print("DỮ LIỆU GỐC:")
print(ds)

print("\nDỮ LIỆU ĐÃ TÁCH TỪ:")
print(segmented)

print("\nVí dụ:")
print(segmented["train"][0])


In [ ]:

import shutil
import os

src = os.path.join(INPUT_DIR, "results")
dst = os.path.join(BASE_DIR, "results")

if os.path.isdir(src):
    shutil.copytree(
        src,
        dst,
        dirs_exist_ok=True
    )
    print("Đã khôi phục kết quả cũ!")
else:
    print("Không tìm thấy results!")

print("Các thư mục:", os.listdir(dst))


In [ ]:

import torch

print("CUDA:", torch.cuda.is_available())
print("Số GPU:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))


In [ ]:
import os, sys, subprocess, torch
print('Python:',sys.version.split()[0]);print('GPU:',torch.cuda.is_available())
if torch.cuda.is_available():print('Thiết bị:',torch.cuda.get_device_name(0))
print('Input:',os.listdir('/kaggle/input'))
# Bật GPU trong Notebook Settings trước khi chạy phần C/D/E/G

In [ ]:
# Kaggle thường đã có numpy/pandas/sklearn/torch. KHÔNG dùng -U pandas/numpy.
# Chỉ bật nếu thiếu module, và cần Internet để pip tải.
RUN_INSTALL=False
if RUN_INSTALL:
    %pip install -q datasets transformers accelerate bitsandbytes safetensors sentencepiece py_vncorenlp scipy scikit-learn
else:
    print('Bỏ qua pip. Nếu import ở Cell A1 lỗi, bật RUN_INSTALL rồi chạy lại.')


In [ ]:

import os
import glob
import shutil
import zipfile

# Dataset đã được Kaggle giải nén
INPUT = (
    "/kaggle/input/datasets/embehacker/"
    "vsfc-experiment-backup"
)

BASE = "/kaggle/working/vsfc_llm"
DATA = os.path.join(BASE, "data")
SEG = os.path.join(BASE, "segmented_data")
OUT = os.path.join(BASE, "results", "remaining")

assert os.path.isdir(INPUT), (
    f"Không tìm thấy Kaggle Dataset: {INPUT}"
)

os.makedirs(BASE, exist_ok=True)

# Sao chép dữ liệu gốc và đã tách từ
for folder in ["data", "segmented_data", "results"]:
    src = os.path.join(INPUT, folder)
    dst = os.path.join(BASE, folder)

    if os.path.isdir(src):
        shutil.copytree(
            src, dst,
            dirs_exist_ok=True
        )
        print("Đã khôi phục:", folder)
    else:
        print("Không tìm thấy:", folder)

os.makedirs(OUT, exist_ok=True)

print("\nBASE:", BASE)
print("DATA:", DATA)
print("SEG:", SEG)
print("OUT:", OUT)
print("\nKHÔI PHỤC THÀNH CÔNG!")


In [ ]:

from datasets import load_from_disk

ds = load_from_disk(DATA)
segmented = load_from_disk(SEG)

print("=== DỮ LIỆU GỐC ===")
print(ds)

print("\n=== DỮ LIỆU ĐÃ TÁCH TỪ ===")
print(segmented)

print("\nVí dụ:")
print(segmented["train"][0])


In [ ]:

import torch

print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("data_for_task:",
      "data_for_task" in globals())

print("run_phobert:",
      "run_phobert" in globals())

print("OUT:", OUT)


In [ ]:

import os, math, time, gc, inspect
import numpy as np
import pandas as pd
import torch

from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    set_seed
)
from sklearn.metrics import accuracy_score, f1_score

SEG = "/kaggle/working/vsfc_llm/segmented_data"
OUT = "/kaggle/working/vsfc_llm/results/remaining"
MODEL_NAME = "vinai/phobert-base-v2"

os.makedirs(OUT, exist_ok=True)

assert torch.cuda.is_available(), "Chưa có GPU!"
assert os.path.exists(SEG), "Thiếu segmented_data!"

seg = load_from_disk(SEG)

tok = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False
)

segtok = seg.map(
    lambda batch: tok(
        batch["segmented_text"],
        truncation=True,
        max_length=128
    ),
    batched=True
)

collator = DataCollatorWithPadding(tokenizer=tok)

def data_for_task(split, task):
    d = segtok[split]
    if "labels" in d.column_names:
        d = d.remove_columns("labels")
    d = d.rename_column(task, "labels")
    return d.remove_columns([
        c for c in d.column_names
        if c not in ("input_ids", "attention_mask", "labels")
    ])

def metricfn(result):
    logits, labels = result
    pred = np.argmax(logits, axis=-1)
    return {
        "macro_f1": f1_score(
            labels, pred, average="macro", zero_division=0
        ),
        "accuracy": accuracy_score(labels, pred),
        "weighted_f1": f1_score(
            labels, pred, average="weighted", zero_division=0
        )
    }

print("GPU:", torch.cuda.get_device_name(0))
print("ĐÃ CHUẨN BỊ DỮ LIỆU PHOBERT!")


In [ ]:

import requests
import socket

try:
    print(
        "DNS:",
        socket.gethostbyname("huggingface.co")
    )

    response = requests.get(
        "https://huggingface.co",
        timeout=10
    )

    print("HTTP:", response.status_code)

except Exception as e:
    print("LỖI KẾT NỐI:", repr(e))


In [ ]:

import torch

print("CUDA:", torch.cuda.is_available())
print("Số GPU:", torch.cuda.device_count())

if torch.cuda.is_available():
    x = torch.randn(3000, 3000, device="cuda")
    y = x @ x
    torch.cuda.synchronize()

    print("Tính toán GPU thành công!")
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory (MB):",
        round(torch.cuda.memory_allocated() / 1024**2)
    )

    del x, y
    torch.cuda.empty_cache()


In [ ]:

import os, time, gc, math, inspect
import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    set_seed
)
from sklearn.metrics import accuracy_score, f1_score

MODEL_NAME = "vinai/phobert-base-v2"
OUT = "/kaggle/working/vsfc_llm/results/remaining"
os.makedirs(OUT, exist_ok=True)

def metricfn(eval_pred):
    logits, labels = eval_pred
    pred = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, pred),
        "macro_f1": f1_score(
            labels, pred, average="macro",
            zero_division=0
        ),
        "weighted_f1": f1_score(
            labels, pred, average="weighted",
            zero_division=0
        )
    }

def run_phobert_topic(seed):
    tag = f"topic_full_seed_{seed}"
    folder = os.path.join(OUT, tag)
    os.makedirs(folder, exist_ok=True)

    metrics_file = os.path.join(folder, "metrics.csv")
    pred_file = os.path.join(folder, "pred.csv")

    if os.path.exists(metrics_file) and os.path.exists(pred_file):
        print("Đã hoàn thành:", tag)
        return pd.read_csv(metrics_file).iloc[0].to_dict()

    assert torch.cuda.is_available(), "Chưa có GPU!"
    assert callable(globals().get("data_for_task")), (
        "Chưa chạy Cell tạo data_for_task()"
    )

    set_seed(seed)

    tr = data_for_task("train", "topic")
    dv = data_for_task("validation", "topic")
    te = data_for_task("test", "topic")

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=4
    )

    params = inspect.signature(
        TrainingArguments.__init__
    ).parameters

    cfg = {
        "output_dir": os.path.join(folder, "checkpoints"),
        "learning_rate": 2e-5,
        "per_device_train_batch_size": 8,
        "per_device_eval_batch_size": 16,
        "gradient_accumulation_steps": 4,
        "num_train_epochs": 5,
        "weight_decay": 0.01,
        "fp16": True,
        "save_strategy": "epoch",
        "load_best_model_at_end": True,
        "metric_for_best_model": "macro_f1",
        "greater_is_better": True,
        "save_total_limit": 2,
        "logging_steps": 50,
        "report_to": "none",
        "seed": seed,
        "data_seed": seed
    }

    if "eval_strategy" in params:
        cfg["eval_strategy"] = "epoch"
    else:
        cfg["evaluation_strategy"] = "epoch"

    if "warmup_ratio" in params:
        cfg["warmup_ratio"] = 0.1
    else:
        steps = math.ceil(math.ceil(len(tr) / 8) / 4)
        cfg["warmup_steps"] = math.ceil(steps * 5 * 0.1)

    args = TrainingArguments(**cfg)

    trainer_kwargs = {
        "model": model,
        "args": args,
        "train_dataset": tr,
        "eval_dataset": dv,
        "data_collator": DataCollatorWithPadding(
            tokenizer=tok
        ),
        "compute_metrics": metricfn,
        "callbacks": [
            EarlyStoppingCallback(
                early_stopping_patience=3
            )
        ]
    }

    trainer_params = inspect.signature(
        Trainer.__init__
    ).parameters

    if "processing_class" in trainer_params:
        trainer_kwargs["processing_class"] = tok
    else:
        trainer_kwargs["tokenizer"] = tok

    trainer = Trainer(**trainer_kwargs)

    start = time.time()
    trainer.train()

    output = trainer.predict(te)
    y_pred = np.argmax(output.predictions, axis=-1)
    y_true = np.array(te["labels"])

    result = {
        "task": "topic",
        "tag": tag,
        "seed": seed,
        "train_n": len(tr),
        "dev_n": len(dv),
        "labels_used": len(tr) + len(dv),
        "best_dev_macro_f1": trainer.state.best_metric,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true, y_pred, average="macro"
        ),
        "weighted_f1": f1_score(
            y_true, y_pred, average="weighted"
        ),
        "duration_minutes": round(
            (time.time() - start) / 60, 2
        )
    }

    pd.DataFrame({
        "sample_id": np.arange(len(y_true)),
        "y_true": y_true,
        "y_pred": y_pred
    }).to_csv(pred_file, index=False)

    pd.DataFrame([result]).to_csv(
        metrics_file, index=False
    )

    print("\nHOÀN THÀNH:", tag)
    print("Macro-F1:", result["macro_f1"])
    print("Đã lưu:", folder)

    del trainer, model, output
    gc.collect()
    torch.cuda.empty_cache()

    return result

print("ĐÃ TẠO HÀM run_phobert_topic THÀNH CÔNG!")


In [ ]:

print("GPU:", torch.cuda.is_available())
print("Tokenizer:", "tok" in globals())
print("Dữ liệu:", callable(globals().get("data_for_task")))
print("Hàm huấn luyện:", callable(globals().get("run_phobert_topic")))

result = run_phobert_topic(42)

print("\nKẾT QUẢ SEED 42:")
display(pd.DataFrame([result]))


In [ ]:

print("GPU:", torch.cuda.is_available())
print("Tokenizer:", "tok" in globals())
print("Dữ liệu:", callable(globals().get("data_for_task")))
print("Hàm huấn luyện:", callable(globals().get("run_phobert_topic")))

result = run_phobert_topic(42)

print("\nKẾT QUẢ SEED 42:")
display(pd.DataFrame([result]))


In [ ]:

for seed in [43, 44, 45, 46]:
    print(f"\n===== PHOBERT TOPIC SEED {seed} =====")

    result = run_phobert_topic(seed)

    print(
        f"Seed {seed} hoàn thành, "
        f"Macro-F1 = {result['macro_f1']:.4f}"
    )

print("\nHOÀN THÀNH PHOBERT TOPIC 5 SEED!")


In [ ]:

import os

OUT = "/kaggle/working/vsfc_llm/results/remaining"

for seed in [42, 43, 44, 45, 46]:
    folder = f"{OUT}/topic_full_seed_{seed}"

    metrics = os.path.join(folder, "metrics.csv")
    pred = os.path.join(folder, "pred.csv")

    if os.path.isfile(metrics) and os.path.isfile(pred):
        print(f"Seed {seed}: ĐÃ HOÀN THÀNH")
    else:
        print(f"Seed {seed}: CHƯA CÓ KẾT QUẢ")


In [ ]:

import os
from datasets import load_from_disk
from transformers import DataCollatorWithPadding

INPUT = (
    "/kaggle/input/datasets/embehacker/"
    "vsfc-experiment-backup"
)

BASE = "/kaggle/working/vsfc_llm"
SEG = os.path.join(BASE, "segmented_data")
CACHE_DIR = os.path.join(BASE, "token_cache")

os.makedirs(BASE, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# Sao chép segmented_data sang nơi cho phép ghi
if not os.path.exists(os.path.join(SEG, "dataset_dict.json")):
    import shutil
    shutil.copytree(
        os.path.join(INPUT, "segmented_data"),
        SEG,
        dirs_exist_ok=True
    )

seg = load_from_disk(SEG)

# Chỉ định vị trí cache có quyền ghi
cache_files = {
    split: os.path.join(
        CACHE_DIR, f"{split}_tokenized.arrow"
    )
    for split in ["train", "validation", "test"]
}

segtok = seg.map(
    lambda batch: tok(
        batch["segmented_text"],
        truncation=True,
        max_length=128
    ),
    batched=True,
    cache_file_names=cache_files
)

collator = DataCollatorWithPadding(tokenizer=tok)

def data_for_task(split, task):
    d = segtok[split]

    if "labels" in d.column_names:
        d = d.remove_columns("labels")

    d = d.rename_column(task, "labels")

    return d.remove_columns([
        c for c in d.column_names
        if c not in (
            "input_ids",
            "attention_mask",
            "labels"
        )
    ])

print("TOKENIZE THÀNH CÔNG!")
print("Train:", len(segtok["train"]))
print("Dev:", len(segtok["validation"]))
print("Test:", len(segtok["test"]))
print("Cache:", CACHE_DIR)


In [ ]:

import torch

print("GPU:", torch.cuda.is_available())
print("Tokenizer:", "tok" in globals())
print("Dữ liệu:", callable(globals().get("data_for_task")))
print("Hàm huấn luyện:",
      callable(globals().get("run_phobert_topic")))

tr = data_for_task("train", "topic")

print("Số câu train:", len(tr))
print("Các cột:", tr.column_names)
print("Nhãn mẫu đầu tiên:", tr[0]["labels"])


In [ ]:

import os

OUT = "/kaggle/working/vsfc_llm/results/remaining"

for seed in range(42, 47):
    folder = f"{OUT}/topic_full_seed_{seed}"

    done = (
        os.path.exists(f"{folder}/metrics.csv")
        and os.path.exists(f"{folder}/pred.csv")
    )

    print(
        f"Seed {seed}:",
        "ĐÃ XONG" if done else "CHƯA XONG"
    )


In [ ]:

result = run_phobert_topic(42)

print("\nKẾT QUẢ SEED 42:")
display(pd.DataFrame([result]))


In [ ]:

import os
import pandas as pd

OUT = "/kaggle/working/vsfc_llm/results/remaining"

rows = []

for seed in range(42, 47):
    folder = os.path.join(OUT, f"topic_full_seed_{seed}")
    metrics_file = os.path.join(folder, "metrics.csv")
    pred_file = os.path.join(folder, "pred.csv")

    assert os.path.isfile(metrics_file), f"Thiếu metrics seed {seed}"
    assert os.path.isfile(pred_file), f"Thiếu pred seed {seed}"
    assert len(pd.read_csv(pred_file)) == 3166, (
        f"Seed {seed} chưa dự đoán đủ 3166 câu"
    )

    rows.append(pd.read_csv(metrics_file).iloc[0])

topic_summary = pd.DataFrame(rows)

display(topic_summary)

print("\n=== PHOBERT TOPIC — 5 SEED ===")

for col in ["accuracy", "macro_f1", "weighted_f1"]:
    print(
        f"{col}: "
        f"{topic_summary[col].mean():.4f} "
        f"± {topic_summary[col].std(ddof=1):.4f}"
    )

topic_summary.to_csv(
    os.path.join(OUT, "phobert_topic_5seed_summary.csv"),
    index=False
)

print("ĐÃ LƯU BẢNG TỔNG HỢP!")


In [ ]:

import zipfile
import os

ZIP_PATH = "/kaggle/working/phobert_topic_5seed_backup.zip"

with zipfile.ZipFile(
    ZIP_PATH, "w", zipfile.ZIP_DEFLATED
) as z:
    for seed in range(42, 47):
        folder = os.path.join(OUT, f"topic_full_seed_{seed}")
        for filename in ["metrics.csv", "pred.csv"]:
            path = os.path.join(folder, filename)
            z.write(
                path,
                arcname=f"topic_full_seed_{seed}/{filename}"
            )

    summary_path = os.path.join(
        OUT, "phobert_topic_5seed_summary.csv"
    )
    z.write(
        summary_path,
        arcname="phobert_topic_5seed_summary.csv"
    )

print("ĐÃ SAO LƯU:", ZIP_PATH)


In [ ]:

import os
import torch
import numpy as np
import pandas as pd
import inspect
import math
import gc
import time

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    set_seed
)

BASE = "/kaggle/working/vsfc_llm"
LC_OUT = f"{BASE}/results/remaining/learning_curve_sentiment"
os.makedirs(LC_OUT, exist_ok=True)

assert torch.cuda.is_available(), "Chưa bật GPU"
assert "tok" in globals(), "Chưa khôi phục tokenizer"
assert callable(globals().get("data_for_task")), "Chưa khôi phục dữ liệu"

LC_RATIOS = [1, 2, 5, 10, 25, 50, 100]
LC_SEEDS = [42, 43, 44]

train_all = data_for_task("train", "sentiment")
dev_all = data_for_task("validation", "sentiment")
test_all = data_for_task("test", "sentiment")

def stratified_subset(dataset, n, seed):
    n = min(n, len(dataset))
    if n == len(dataset):
        return dataset

    indices = np.arange(len(dataset))
    labels = np.asarray(dataset["labels"])

    picked, _ = train_test_split(
        indices,
        train_size=n,
        random_state=seed,
        stratify=labels
    )
    return dataset.select(sorted(picked.tolist()))

print("GPU:", torch.cuda.get_device_name(0))
print("Train:", len(train_all))
print("Dev:", len(dev_all))
print("Test:", len(test_all))
print("Tỷ lệ:", LC_RATIOS)
print("Seeds:", LC_SEEDS)
print("SẴN SÀNG LEARNING CURVE!")


In [ ]:

def run_learning_curve(ratio, seed):
    assert ratio in LC_RATIOS
    assert seed in LC_SEEDS

    tag = f"lc_sentiment_{ratio}pct_seed_{seed}"
    folder = os.path.join(LC_OUT, tag)
    os.makedirs(folder, exist_ok=True)

    metrics_file = f"{folder}/metrics.csv"
    pred_file = f"{folder}/pred.csv"

    if os.path.isfile(metrics_file) and os.path.isfile(pred_file):
        print("BỎ QUA, ĐÃ CÓ:", tag)
        return pd.read_csv(metrics_file).iloc[0].to_dict()

    set_seed(seed)

    n_train = max(3, round(len(train_all) * ratio / 100))
    n_dev = max(100, round(len(dev_all) * ratio / 100))

    tr = stratified_subset(train_all, n_train, seed)
    dv = stratified_subset(dev_all, n_dev, seed)

    print(f"\nCHẠY {tag}")
    print(f"Train={len(tr)}, Dev={len(dv)}, Test={len(test_all)}")

    model = AutoModelForSequenceClassification.from_pretrained(
        "vinai/phobert-base-v2",
        num_labels=3
    )

    batch_size = 8
    grad_accum = 4

    updates_per_epoch = math.ceil(
        math.ceil(len(tr) / batch_size) / grad_accum
    )

    # Tập nhỏ cần đủ 300 bước tối ưu
    total_steps = max(300, updates_per_epoch * 5)

    params = inspect.signature(
        TrainingArguments.__init__
    ).parameters

    config = {
        "output_dir": f"{folder}/checkpoints",
        "learning_rate": 2e-5,
        "per_device_train_batch_size": batch_size,
        "per_device_eval_batch_size": 16,
        "gradient_accumulation_steps": grad_accum,
        "max_steps": total_steps,
        "weight_decay": 0.01,
        "fp16": True,
        "save_strategy": "steps",
        "save_steps": max(50, total_steps // 5),
        "eval_steps": max(50, total_steps // 5),
        "load_best_model_at_end": True,
        "metric_for_best_model": "macro_f1",
        "greater_is_better": True,
        "save_total_limit": 1,
        "logging_steps": 25,
        "report_to": "none",
        "seed": seed,
        "data_seed": seed
    }

    if "eval_strategy" in params:
        config["eval_strategy"] = "steps"
    else:
        config["evaluation_strategy"] = "steps"

    if "warmup_ratio" in params:
        config["warmup_ratio"] = 0.1
    else:
        config["warmup_steps"] = round(total_steps * 0.1)

    args = TrainingArguments(**config)

    def compute_metrics(output):
        logits, labels = output
        preds = np.argmax(logits, axis=-1)

        return {
            "accuracy": accuracy_score(labels, preds),
            "macro_f1": f1_score(
                labels, preds,
                average="macro",
                zero_division=0
            )
        }

    kwargs = {
        "model": model,
        "args": args,
        "train_dataset": tr,
        "eval_dataset": dv,
        "data_collator": DataCollatorWithPadding(tokenizer=tok),
        "compute_metrics": compute_metrics,
        "callbacks": [
            EarlyStoppingCallback(
                early_stopping_patience=3
            )
        ]
    }

    trainer_params = inspect.signature(
        Trainer.__init__
    ).parameters

    if "processing_class" in trainer_params:
        kwargs["processing_class"] = tok
    else:
        kwargs["tokenizer"] = tok

    trainer = Trainer(**kwargs)

    start = time.time()
    trainer.train()

    output = trainer.predict(test_all)

    y_true = np.asarray(test_all["labels"])
    y_pred = np.argmax(output.predictions, axis=-1)

    result = {
        "task": "sentiment",
        "ratio_pct": ratio,
        "seed": seed,
        "train_n": len(tr),
        "dev_n": len(dv),
        "labels_used": len(tr) + len(dv),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true, y_pred,
            average="macro",
            zero_division=0
        ),
        "weighted_f1": f1_score(
            y_true, y_pred,
            average="weighted",
            zero_division=0
        ),
        "best_dev_macro_f1": trainer.state.best_metric,
        "duration_minutes": (time.time() - start) / 60
    }

    pd.DataFrame({
        "sample_id": np.arange(len(y_true)),
        "y_true": y_true,
        "y_pred": y_pred
    }).to_csv(pred_file, index=False)

    pd.DataFrame([result]).to_csv(
        metrics_file, index=False
    )

    print("\nHOÀN THÀNH:", tag)
    print("Macro-F1:", result["macro_f1"])
    print("Đã lưu:", folder)

    del trainer, model, output
    gc.collect()
    torch.cuda.empty_cache()

    return result

print("ĐÃ TẠO HÀM LEARNING CURVE!")


In [ ]:

result = run_learning_curve(1, 42)

display(pd.DataFrame([result]))


In [ ]:

import os
import shutil

disk = shutil.disk_usage("/kaggle/working")

print("Tổng ổ đĩa:", round(disk.total / 1024**3, 2), "GB")
print("Đã dùng:", round(disk.used / 1024**3, 2), "GB")
print("Còn trống:", round(disk.free / 1024**3, 2), "GB")

folder = (
    "/kaggle/working/vsfc_llm/results/remaining/"
    "learning_curve_sentiment/lc_sentiment_1pct_seed_42"
)

print("\n=== CHECKPOINT ĐÃ TẠO ===")

if os.path.isdir(folder):
    for root, dirs, files in os.walk(folder):
        if "checkpoint-" in root:
            size = sum(
                os.path.getsize(os.path.join(root, f))
                for f in files
            )
            print(root, round(size / 1024**2, 1), "MB")
else:
    print("Chưa thấy thư mục kết quả")


In [ ]:

import os
import gc
import time
import math
import inspect
import shutil

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    set_seed
)

# ======================================
# 1. CẤU HÌNH
# ======================================

BASE = "/kaggle/working/vsfc_llm"

LC_OUT = os.path.join(
    BASE,
    "results",
    "remaining",
    "learning_curve_sentiment"
)

MODEL_NAME = "vinai/phobert-base-v2"

LC_RATIOS = [1, 2, 5, 10, 25, 50, 100]
LC_SEEDS = [42, 43, 44]

os.makedirs(LC_OUT, exist_ok=True)

assert torch.cuda.is_available(), "Chưa bật GPU"
assert "tok" in globals(), "Chưa có tokenizer"
assert callable(globals().get("data_for_task")), (
    "Chưa khôi phục hàm data_for_task"
)

train_all = data_for_task("train", "sentiment")
dev_all = data_for_task("validation", "sentiment")
test_all = data_for_task("test", "sentiment")

# ======================================
# 2. LẤY MẪU PHÂN TẦNG
# ======================================

def stratified_subset(dataset, n, seed):
    n = min(int(n), len(dataset))

    if n >= len(dataset):
        return dataset

    indices = np.arange(len(dataset))
    labels = np.asarray(dataset["labels"])

    selected, _ = train_test_split(
        indices,
        train_size=n,
        random_state=seed,
        stratify=labels
    )

    return dataset.select(sorted(selected.tolist()))

# ======================================
# 3. ĐỊNH NGHĨA HÀM HUẤN LUYỆN
# ======================================

def run_learning_curve(ratio, seed):

    assert ratio in LC_RATIOS
    assert seed in LC_SEEDS

    tag = f"lc_sentiment_{ratio}pct_seed_{seed}"

    folder = os.path.join(LC_OUT, tag)
    os.makedirs(folder, exist_ok=True)

    metrics_file = os.path.join(folder, "metrics.csv")
    pred_file = os.path.join(folder, "pred.csv")

    # Nếu đã đủ file thì không huấn luyện lại
    if (
        os.path.isfile(metrics_file)
        and os.path.isfile(pred_file)
        and len(pd.read_csv(pred_file)) == len(test_all)
    ):
        print("ĐÃ HOÀN THÀNH, BỎ QUA:", tag)
        return pd.read_csv(metrics_file).iloc[0].to_dict()

    # Kiểm tra dung lượng trước khi chạy
    free_gb = shutil.disk_usage(
        "/kaggle/working"
    ).free / 1024**3

    print(f"Ổ đĩa còn trống: {free_gb:.2f} GB")

    if free_gb < 2:
        raise RuntimeError(
            "Ổ đĩa còn dưới 2 GB. "
            "Hãy giải phóng dung lượng trước."
        )

    set_seed(seed)

    n_train = max(
        3, round(len(train_all) * ratio / 100)
    )

    n_dev = max(
        100, round(len(dev_all) * ratio / 100)
    )

    tr = stratified_subset(
        train_all, n_train, seed
    )

    dv = stratified_subset(
        dev_all, n_dev, seed
    )

    print("\n==============================")
    print("CHẠY:", tag)
    print("Train:", len(tr))
    print("Dev:", len(dv))
    print("Test:", len(test_all))
    print("==============================")

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=3
    )

    batch_size = 8
    grad_accum = 4

    updates_per_epoch = math.ceil(
        math.ceil(len(tr) / batch_size) / grad_accum
    )

    total_steps = max(
        300,
        updates_per_epoch * 5
    )

    params = inspect.signature(
        TrainingArguments.__init__
    ).parameters

    config = {
        "output_dir": os.path.join(
            folder, "checkpoints"
        ),
        "learning_rate": 2e-5,
        "per_device_train_batch_size": batch_size,
        "per_device_eval_batch_size": 16,
        "gradient_accumulation_steps": grad_accum,
        "max_steps": total_steps,
        "weight_decay": 0.01,
        "fp16": True,
        "save_strategy": "steps",
        "save_steps": max(50, total_steps // 5),
        "eval_steps": max(50, total_steps // 5),
        "load_best_model_at_end": True,
        "metric_for_best_model": "macro_f1",
        "greater_is_better": True,
        "save_total_limit": 1,
        "logging_steps": 25,
        "report_to": "none",
        "seed": seed,
        "data_seed": seed
    }

    if "eval_strategy" in params:
        config["eval_strategy"] = "steps"
    else:
        config["evaluation_strategy"] = "steps"

    if "warmup_ratio" in params:
        config["warmup_ratio"] = 0.1
    else:
        config["warmup_steps"] = round(
            total_steps * 0.1
        )

    # QUAN TRỌNG:
    # Chỉ lưu trọng số model,
    # không lưu optimizer và scheduler
    if "save_only_model" in params:
        config["save_only_model"] = True
    else:
        raise RuntimeError(
            "Transformers chưa hỗ trợ save_only_model"
        )

    args = TrainingArguments(**config)

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        return {
            "accuracy": accuracy_score(
                labels, preds
            ),
            "macro_f1": f1_score(
                labels,
                preds,
                average="macro",
                zero_division=0
            ),
            "weighted_f1": f1_score(
                labels,
                preds,
                average="weighted",
                zero_division=0
            )
        }

    kwargs = {
        "model": model,
        "args": args,
        "train_dataset": tr,
        "eval_dataset": dv,
        "data_collator": DataCollatorWithPadding(
            tokenizer=tok
        ),
        "compute_metrics": compute_metrics,
        "callbacks": [
            EarlyStoppingCallback(
                early_stopping_patience=3
            )
        ]
    }

    trainer_params = inspect.signature(
        Trainer.__init__
    ).parameters

    if "processing_class" in trainer_params:
        kwargs["processing_class"] = tok
    else:
        kwargs["tokenizer"] = tok

    trainer = Trainer(**kwargs)

    start = time.time()

    trainer.train()

    output = trainer.predict(test_all)

    y_true = np.asarray(
        test_all["labels"]
    )

    y_pred = np.argmax(
        output.predictions,
        axis=-1
    )

    result = {
        "task": "sentiment",
        "ratio_pct": ratio,
        "seed": seed,
        "train_n": len(tr),
        "dev_n": len(dv),
        "labels_used": len(tr) + len(dv),
        "accuracy": accuracy_score(
            y_true, y_pred
        ),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "weighted_f1": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        ),
        "best_dev_macro_f1": trainer.state.best_metric,
        "duration_minutes": (
            time.time() - start
        ) / 60
    }

    # Lưu dự đoán và chỉ số đánh giá
    pd.DataFrame({
        "sample_id": np.arange(
            len(y_true)
        ),
        "y_true": y_true,
        "y_pred": y_pred
    }).to_csv(
        pred_file,
        index=False
    )

    pd.DataFrame([
        result
    ]).to_csv(
        metrics_file,
        index=False
    )

    print("\nHOÀN THÀNH:", tag)
    print(
        "Macro-F1:",
        round(result["macro_f1"], 4)
    )
    print("Đã lưu:", folder)

    # Xóa checkpoint sau khi đã lưu
    # metrics.csv và pred.csv thành công
    del trainer, model, output
    gc.collect()
    torch.cuda.empty_cache()

    checkpoints = os.path.join(
        folder, "checkpoints"
    )

    if os.path.isdir(checkpoints):
        shutil.rmtree(checkpoints)
        print("Đã dọn checkpoint để tiết kiệm ổ đĩa")

    return result

print("\nĐÃ TẠO HÀM LEARNING CURVE THÀNH CÔNG!")
print("GPU:", torch.cuda.is_available())
print("Train:", len(train_all))
print("Dev:", len(dev_all))
print("Test:", len(test_all))
print("Hàm:", callable(globals().get("run_learning_curve")))


In [ ]:

result = run_learning_curve(1, 42)

display(pd.DataFrame([result]))


In [ ]:

import pandas as pd

for seed in [43, 44]:
    print(f"\n===== LEARNING CURVE 1% - SEED {seed} =====")

    result = run_learning_curve(1, seed)

    print(
        f"Seed {seed} | "
        f"Macro-F1 = {result['macro_f1']:.4f}"
    )

print("\nĐÃ HOÀN THÀNH 1% VỚI 3 SEED!")


In [ ]:

import os
import pandas as pd

rows = []

for seed in [42, 43, 44]:
    path = os.path.join(
        LC_OUT,
        f"lc_sentiment_1pct_seed_{seed}",
        "metrics.csv"
    )

    assert os.path.isfile(path), f"Thiếu seed {seed}"

    rows.append(pd.read_csv(path).iloc[0])

summary_1pct = pd.DataFrame(rows)

display(summary_1pct)

print(
    "\nMacro-F1 trung bình 1%:",
    round(summary_1pct["macro_f1"].mean(), 4)
)

print(
    "Độ lệch chuẩn:",
    round(summary_1pct["macro_f1"].std(ddof=1), 4)
)


In [ ]:

import shutil

backup_path = shutil.make_archive(
    "/kaggle/working/learning_curve_backup",
    "zip",
    root_dir=LC_OUT
)

print("ĐÃ TẠO BACKUP:", backup_path)


In [ ]:

import os
import pandas as pd

LC_OUT = (
    "/kaggle/working/vsfc_llm/results/remaining/"
    "learning_curve_sentiment"
)

completed = []
missing = []

for ratio in [1, 2, 5, 10, 25, 50, 100]:
    for seed in [42, 43, 44]:
        folder = f"{LC_OUT}/lc_sentiment_{ratio}pct_seed_{seed}"
        metrics = f"{folder}/metrics.csv"
        pred = f"{folder}/pred.csv"

        if os.path.isfile(metrics) and os.path.isfile(pred):
            completed.append((ratio, seed))
        else:
            missing.append((ratio, seed))

print(f"ĐÃ HOÀN THÀNH: {len(completed)}/21")
print("Các lượt đã xong:", completed)
print("Các lượt còn thiếu:", missing)


In [ ]:

for seed in [42, 43, 44]:
    result = run_learning_curve(2, seed)

    print(
        f"2% | Seed {seed} | "
        f"Macro-F1 = {result['macro_f1']:.4f}"
    )


In [ ]:
print(f"Đã hoàn thành: {len(completed)}/21")
print("Còn thiếu:", missing)

In [ ]:

for seed in [42, 43, 44]:
    result = run_learning_curve(2, seed)
    print(f"2% - Seed {seed}: Macro-F1 = {result['macro_f1']:.4f}")


In [ ]:

import os
import pandas as pd

RATIOS = [5, 10, 25, 50, 100]
SEEDS = [42, 43, 44]

for ratio in RATIOS:
    print(f"\n{'='*45}")
    print(f"LEARNING CURVE {ratio}%")
    print(f"{'='*45}")

    for seed in SEEDS:
        folder = os.path.join(
            LC_OUT,
            f"lc_sentiment_{ratio}pct_seed_{seed}"
        )

        metrics_file = os.path.join(folder, "metrics.csv")
        pred_file = os.path.join(folder, "pred.csv")

        if os.path.isfile(metrics_file) and os.path.isfile(pred_file):
            if len(pd.read_csv(pred_file)) == 3166:
                print(f"BỎ QUA {ratio}% seed {seed}: đã hoàn thành")
                continue

        result = run_learning_curve(ratio, seed)

        print(
            f"HOÀN THÀNH {ratio}% seed {seed} | "
            f"Macro-F1 = {result['macro_f1']:.4f}"
        )

    print(f"\nĐÃ XONG NHÓM {ratio}%")


In [ ]:

import os
import pandas as pd

LC_OUT = (
    "/kaggle/working/vsfc_llm/results/remaining/"
    "learning_curve_sentiment"
)

completed = []
missing = []

for ratio in [1, 2, 5, 10, 25, 50, 100]:
    for seed in [42, 43, 44]:
        folder = f"{LC_OUT}/lc_sentiment_{ratio}pct_seed_{seed}"
        mp = f"{folder}/metrics.csv"
        pp = f"{folder}/pred.csv"

        if (
            os.path.isfile(mp)
            and os.path.isfile(pp)
            and len(pd.read_csv(pp)) == 3166
        ):
            completed.append((ratio, seed))
        else:
            missing.append((ratio, seed))

print(f"HOÀN THÀNH: {len(completed)}/21")
print("Đã xong:", completed)
print("Còn thiếu:", missing)


In [ ]:

import os
import pandas as pd

remaining = [
    (100, 43),
    (100, 44)
]

for ratio, seed in remaining:
    print(f"\n===== LEARNING CURVE {ratio}% - SEED {seed} =====")

    result = run_learning_curve(ratio, seed)

    print(
        f"HOÀN THÀNH {ratio}% seed {seed} | "
        f"Macro-F1 = {result['macro_f1']:.4f}"
    )

print("\nĐÃ XỬ LÝ 4 LƯỢT CUỐI!")


In [ ]:

import os
import shutil
import pandas as pd

LC_OUT = "/kaggle/working/vsfc_llm/results/remaining/learning_curve_sentiment"

completed = []
missing = []

for ratio in [1, 2, 5, 10, 25, 50, 100]:
    for seed in [42, 43, 44]:
        folder = f"{LC_OUT}/lc_sentiment_{ratio}pct_seed_{seed}"
        mp = f"{folder}/metrics.csv"
        pp = f"{folder}/pred.csv"

        if (
            os.path.isfile(mp)
            and os.path.isfile(pp)
            and len(pd.read_csv(pp)) == 3166
        ):
            completed.append((ratio, seed))
        else:
            missing.append((ratio, seed))

print(f"HOÀN THÀNH: {len(completed)}/21")
print("CÒN THIẾU:", missing)

backup = shutil.make_archive(
    "/kaggle/working/learning_curve_BACKUP_MOI_NHAT",
    "zip",
    root_dir=LC_OUT
)

print("BACKUP:", backup)


In [ ]:

import os
from IPython.display import display, HTML

path = "/kaggle/working/learning_curve_BACKUP_MOI_NHAT.zip"

assert os.path.isfile(path), "File ZIP không tồn tại!"

size = os.path.getsize(path) / 1024**2
print(f"File tồn tại: {size:.2f} MB")

display(HTML(
    '<a href="/files/learning_curve_BACKUP_MOI_NHAT.zip" '
    'target="_blank" download>'
    'BẤM ĐỂ TẢI FILE BACKUP'
    '</a>'
))


In [ ]:

import os
import glob

BASE = "/kaggle/working/vsfc_llm"
LC_OUT = f"{BASE}/results/remaining/learning_curve_sentiment"

print("=== KIỂM TRA DỮ LIỆU ===")
print("Segmented data:",
      os.path.exists(f"{BASE}/segmented_data/dataset_dict.json"))
print("Thư mục Learning Curve:", os.path.isdir(LC_OUT))

print("\n=== KIỂM TRA KẾT QUẢ ===")
completed = []
missing = []

for ratio in [1, 2, 5, 10, 25, 50, 100]:
    for seed in [42, 43, 44]:
        folder = f"{LC_OUT}/lc_sentiment_{ratio}pct_seed_{seed}"
        if (os.path.isfile(f"{folder}/metrics.csv")
            and os.path.isfile(f"{folder}/pred.csv")):
            completed.append((ratio, seed))
        else:
            missing.append((ratio, seed))

print(f"Đã có: {len(completed)}/21")
print("Còn thiếu:", missing)

print("\n=== FILE BACKUP ĐANG CÓ TRONG KAGGLE ===")
for path in glob.glob("/kaggle/input/**/*.zip", recursive=True):
    print(path)

for path in glob.glob("/kaggle/working/*.zip"):
    print(path)


In [ ]:

import os

print("=== DATASET ĐÃ GẮN ===")

for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)

    if level > 3:
        dirs[:] = []
        continue

    print(root)

    for file in files[:8]:
        print("   ", file)


In [ ]:

import os
import glob
import zipfile
import shutil

LC_OUT = (
    "/kaggle/working/vsfc_llm/results/remaining/"
    "learning_curve_sentiment"
)

os.makedirs(LC_OUT, exist_ok=True)

# Tìm ZIP backup
zips = glob.glob(
    "/kaggle/input/**/*learning_curve*BACKUP*.zip",
    recursive=True
)

restored = 0

if zips:
    zip_path = zips[0]
    print("Tìm thấy ZIP:", zip_path)

    with zipfile.ZipFile(zip_path) as z:
        for info in z.infolist():
            if info.is_dir():
                continue

            name = info.filename.replace("\\", "/")

            if name.startswith("/") or ".." in name.split("/"):
                raise ValueError("ZIP chứa đường dẫn không an toàn")

            destination = os.path.join(LC_OUT, name)

            os.makedirs(
                os.path.dirname(destination),
                exist_ok=True
            )

            if not os.path.exists(destination):
                with z.open(info) as source:
                    with open(destination, "wb") as target:
                        shutil.copyfileobj(source, target)

                restored += 1

else:
    # Kaggle đã tự giải nén ZIP
    metrics_files = glob.glob(
        "/kaggle/input/**/lc_sentiment_*pct_seed_*/metrics.csv",
        recursive=True
    )

    if not metrics_files:
        raise FileNotFoundError(
            "Chưa tìm thấy backup! Kiểm tra Add Input."
        )

    for metrics in metrics_files:
        source_folder = os.path.dirname(metrics)
        name = os.path.basename(source_folder)
        target_folder = os.path.join(LC_OUT, name)

        os.makedirs(target_folder, exist_ok=True)

        for filename in ["metrics.csv", "pred.csv"]:
            src = os.path.join(source_folder, filename)
            dst = os.path.join(target_folder, filename)

            if os.path.isfile(src) and not os.path.exists(dst):
                shutil.copy2(src, dst)
                restored += 1

print("Đã khôi phục:", restored, "file")

# Kiểm tra tiến độ
import pandas as pd

completed = []
missing = []

for ratio in [1, 2, 5, 10, 25, 50, 100]:
    for seed in [42, 43, 44]:
        folder = f"{LC_OUT}/lc_sentiment_{ratio}pct_seed_{seed}"

        mp = f"{folder}/metrics.csv"
        pp = f"{folder}/pred.csv"

        if (
            os.path.isfile(mp)
            and os.path.isfile(pp)
            and len(pd.read_csv(pp)) == 3166
        ):
            completed.append((ratio, seed))
        else:
            missing.append((ratio, seed))

print(f"\nĐÃ KHÔI PHỤC: {len(completed)}/21")
print("CÒN THIẾU:", missing)


In [ ]:

import torch

print("GPU:", torch.cuda.is_available())
print("Tokenizer:", "tok" in globals())
print("Dữ liệu:", callable(globals().get("data_for_task")))
print("Hàm Learning Curve:",
      callable(globals().get("run_learning_curve")))


In [ ]:

import os
import shutil
import torch

from datasets import load_from_disk
from transformers import AutoTokenizer, DataCollatorWithPadding

BASE = "/kaggle/working/vsfc_llm"
INPUT = "/kaggle/input/datasets/embehacker/vsfc-experiment-backup"
SEG = f"{BASE}/segmented_data"
LC_OUT = f"{BASE}/results/remaining/learning_curve_sentiment"

assert torch.cuda.is_available(), "Chưa bật GPU!"

# Kiểm tra backup trước khi huấn luyện
import pandas as pd

completed = []
missing = []

for ratio in [1, 2, 5, 10, 25, 50, 100]:
    for seed in [42, 43, 44]:
        folder = f"{LC_OUT}/lc_sentiment_{ratio}pct_seed_{seed}"
        mp = f"{folder}/metrics.csv"
        pp = f"{folder}/pred.csv"

        if (
            os.path.isfile(mp)
            and os.path.isfile(pp)
            and len(pd.read_csv(pp)) == 3166
        ):
            completed.append((ratio, seed))
        else:
            missing.append((ratio, seed))

print(f"ĐÃ KHÔI PHỤC: {len(completed)}/21")
print("CÒN THIẾU:", missing)

assert len(completed) >= 19, (
    "Backup chưa khôi phục đủ 19 lượt. "
    "Đừng huấn luyện tiếp."
)

# Khôi phục segmented_data vào thư mục cho phép ghi
os.makedirs(BASE, exist_ok=True)

if not os.path.isfile(f"{SEG}/dataset_dict.json"):
    source = f"{INPUT}/segmented_data"

    assert os.path.isfile(
        f"{source}/dataset_dict.json"
    ), "Chưa gắn dataset vsfc-experiment-backup!"

    shutil.copytree(source, SEG, dirs_exist_ok=True)

seg = load_from_disk(SEG)

tok = AutoTokenizer.from_pretrained(
    "vinai/phobert-base-v2",
    use_fast=False
)

CACHE = f"{BASE}/token_cache"
os.makedirs(CACHE, exist_ok=True)

segtok = seg.map(
    lambda batch: tok(
        batch["segmented_text"],
        truncation=True,
        max_length=128
    ),
    batched=True,
    cache_file_names={
        split: f"{CACHE}/{split}_tokenized.arrow"
        for split in ["train", "validation", "test"]
    }
)

collator = DataCollatorWithPadding(tokenizer=tok)

def data_for_task(split, task):
    d = segtok[split]

    if "labels" in d.column_names:
        d = d.remove_columns("labels")

    d = d.rename_column(task, "labels")

    return d.remove_columns([
        c for c in d.column_names
        if c not in (
            "input_ids",
            "attention_mask",
            "labels"
        )
    ])

print("\nKHÔI PHỤC MÔI TRƯỜNG THÀNH CÔNG!")
print("GPU:", torch.cuda.is_available())
print("Tokenizer:", "tok" in globals())
print("Dữ liệu:", callable(data_for_task))
print("Train:", len(segtok["train"]))
print("Dev:", len(segtok["validation"]))
print("Test:", len(segtok["test"]))


In [ ]:

import os
import gc
import time
import math
import inspect
import shutil
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    set_seed
)

MODEL_NAME = "vinai/phobert-base-v2"
LC_RATIOS = [1, 2, 5, 10, 25, 50, 100]
LC_SEEDS = [42, 43, 44]

train_all = data_for_task("train", "sentiment")
dev_all = data_for_task("validation", "sentiment")
test_all = data_for_task("test", "sentiment")

os.makedirs(LC_OUT, exist_ok=True)

def stratified_subset(dataset, n, seed):
    n = min(int(n), len(dataset))

    if n == len(dataset):
        return dataset

    indices = np.arange(len(dataset))
    labels = np.asarray(dataset["labels"])

    selected, _ = train_test_split(
        indices,
        train_size=n,
        random_state=seed,
        stratify=labels
    )

    return dataset.select(sorted(selected.tolist()))


def run_learning_curve(ratio, seed):

    assert ratio in LC_RATIOS
    assert seed in LC_SEEDS

    tag = f"lc_sentiment_{ratio}pct_seed_{seed}"
    folder = os.path.join(LC_OUT, tag)

    metrics_file = f"{folder}/metrics.csv"
    pred_file = f"{folder}/pred.csv"

    if (
        os.path.isfile(metrics_file)
        and os.path.isfile(pred_file)
        and len(pd.read_csv(pred_file)) == len(test_all)
    ):
        print("ĐÃ XONG, BỎ QUA:", tag)
        return pd.read_csv(metrics_file).iloc[0].to_dict()

    free_gb = shutil.disk_usage(
        "/kaggle/working"
    ).free / 1024**3

    print(f"Dung lượng trống: {free_gb:.2f} GB")

    if free_gb < 2.5:
        raise RuntimeError(
            "Ổ đĩa còn dưới 2.5 GB. "
            "Cần dọn dung lượng trước."
        )

    os.makedirs(folder, exist_ok=True)
    set_seed(seed)

    n_train = max(
        3, round(len(train_all) * ratio / 100)
    )
    n_dev = max(
        100, round(len(dev_all) * ratio / 100)
    )

    tr = stratified_subset(train_all, n_train, seed)
    dv = stratified_subset(dev_all, n_dev, seed)

    print(f"\n===== CHẠY {tag} =====")
    print(
        f"Train={len(tr)}, "
        f"Dev={len(dv)}, "
        f"Test={len(test_all)}"
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=3
    )

    batch_size = 8
    grad_accum = 4

    updates_per_epoch = math.ceil(
        math.ceil(len(tr) / batch_size) / grad_accum
    )

    total_steps = max(300, updates_per_epoch * 5)
    interval = max(50, math.ceil(total_steps / 5))

    params = inspect.signature(
        TrainingArguments.__init__
    ).parameters

    config = {
        "output_dir": f"{folder}/checkpoints",
        "learning_rate": 2e-5,
        "per_device_train_batch_size": batch_size,
        "per_device_eval_batch_size": 16,
        "gradient_accumulation_steps": grad_accum,
        "max_steps": total_steps,
        "weight_decay": 0.01,
        "fp16": True,
        "save_strategy": "steps",
        "save_steps": interval,
        "eval_steps": interval,
        "load_best_model_at_end": True,
        "metric_for_best_model": "macro_f1",
        "greater_is_better": True,
        "save_total_limit": 1,
        "logging_steps": 25,
        "report_to": "none",
        "seed": seed,
        "data_seed": seed
    }

    if "eval_strategy" in params:
        config["eval_strategy"] = "steps"
    else:
        config["evaluation_strategy"] = "steps"

    if "warmup_ratio" in params:
        config["warmup_ratio"] = 0.1
    else:
        config["warmup_steps"] = round(total_steps * 0.1)

    # Tránh lỗi đầy ổ đĩa hôm trước
    assert "save_only_model" in params, (
        "Transformers chưa hỗ trợ save_only_model"
    )
    config["save_only_model"] = True

    args = TrainingArguments(**config)

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        pred = np.argmax(logits, axis=-1)

        return {
            "accuracy": accuracy_score(labels, pred),
            "macro_f1": f1_score(
                labels,
                pred,
                average="macro",
                zero_division=0
            ),
            "weighted_f1": f1_score(
                labels,
                pred,
                average="weighted",
                zero_division=0
            )
        }

    kwargs = {
        "model": model,
        "args": args,
        "train_dataset": tr,
        "eval_dataset": dv,
        "data_collator": DataCollatorWithPadding(
            tokenizer=tok
        ),
        "compute_metrics": compute_metrics,
        "callbacks": [
            EarlyStoppingCallback(
                early_stopping_patience=3
            )
        ]
    }

    trainer_params = inspect.signature(
        Trainer.__init__
    ).parameters

    if "processing_class" in trainer_params:
        kwargs["processing_class"] = tok
    else:
        kwargs["tokenizer"] = tok

    trainer = Trainer(**kwargs)

    start = time.time()
    trainer.train()

    output = trainer.predict(test_all)

    y_true = np.asarray(test_all["labels"])
    y_pred = np.argmax(output.predictions, axis=-1)

    result = {
        "task": "sentiment",
        "ratio_pct": ratio,
        "seed": seed,
        "train_n": len(tr),
        "dev_n": len(dv),
        "labels_used": len(tr) + len(dv),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true, y_pred,
            average="macro",
            zero_division=0
        ),
        "weighted_f1": f1_score(
            y_true, y_pred,
            average="weighted",
            zero_division=0
        ),
        "best_dev_macro_f1": trainer.state.best_metric,
        "duration_minutes": (
            time.time() - start
        ) / 60
    }

    pd.DataFrame({
        "sample_id": np.arange(len(y_true)),
        "y_true": y_true,
        "y_pred": y_pred
    }).to_csv(pred_file, index=False)

    pd.DataFrame([result]).to_csv(
        metrics_file, index=False
    )

    print("\nHOÀN THÀNH:", tag)
    print("Macro-F1:", result["macro_f1"])

    del trainer, model, output
    gc.collect()
    torch.cuda.empty_cache()

    checkpoints = f"{folder}/checkpoints"

    if os.path.isdir(checkpoints):
        shutil.rmtree(checkpoints)
        print("Đã dọn checkpoint")

    return result


print("ĐÃ KHÔI PHỤC HÀM LEARNING CURVE!")
print(
    "Hàm:",
    callable(globals().get("run_learning_curve"))
)


In [ ]:

assert torch.cuda.is_available()
assert callable(globals().get("run_learning_curve"))

for seed in [43, 44]:
    print(f"\n===== LEARNING CURVE 100% SEED {seed} =====")

    result = run_learning_curve(100, seed)

    print(
        f"Seed {seed} | "
        f"Macro-F1 = {result['macro_f1']:.4f}"
    )


In [ ]:

import os
import pandas as pd

LC_OUT = "/kaggle/working/vsfc_llm/results/remaining/learning_curve_sentiment"

completed = []
missing = []

for ratio in [1, 2, 5, 10, 25, 50, 100]:
    for seed in [42, 43, 44]:
        folder = f"{LC_OUT}/lc_sentiment_{ratio}pct_seed_{seed}"
        mp = f"{folder}/metrics.csv"
        pp = f"{folder}/pred.csv"

        valid = (
            os.path.isfile(mp)
            and os.path.isfile(pp)
        )

        if valid:
            try:
                metrics = pd.read_csv(mp)
                preds = pd.read_csv(pp)
                valid = (
                    len(metrics) == 1
                    and len(preds) == 3166
                    and {"y_true", "y_pred"}.issubset(preds.columns)
                )
            except Exception:
                valid = False

        if valid:
            completed.append((ratio, seed))
        else:
            missing.append((ratio, seed))

print("=" * 45)
print(f"HOÀN THÀNH: {len(completed)}/21")
print("CÒN THIẾU:", missing)
print("=" * 45)


In [ ]:

assert len(completed) == 21, "Chưa đủ 21 lượt!"

rows = []

for ratio, seed in completed:
    path = (
        f"{LC_OUT}/"
        f"lc_sentiment_{ratio}pct_seed_{seed}/metrics.csv"
    )
    rows.append(pd.read_csv(path).iloc[0].to_dict())

all_runs = pd.DataFrame(rows)

summary = (
    all_runs.groupby("ratio_pct", as_index=False)
    .agg(
        train_n=("train_n", "mean"),
        dev_n=("dev_n", "mean"),
        labels_used=("labels_used", "mean"),
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_std=("macro_f1", "std"),
        accuracy_mean=("accuracy", "mean")
    )
    .sort_values("ratio_pct")
)

all_runs.to_csv(
    f"{LC_OUT}/learning_curve_all_runs.csv",
    index=False
)

summary.to_csv(
    f"{LC_OUT}/learning_curve_summary.csv",
    index=False
)

display(summary)

print("ĐÃ LƯU 2 FILE CSV TỔNG HỢP!")


In [ ]:

import shutil
import os
import zipfile

backup_path = shutil.make_archive(
    "/kaggle/working/learning_curve_E3_COMPLETE",
    "zip",
    root_dir=LC_OUT
)

with zipfile.ZipFile(backup_path) as z:
    count = len(z.namelist())
    bad = z.testzip()

print("BACKUP:", backup_path)
print("Số mục trong ZIP:", count)
print("Kiểm tra ZIP:", "OK" if bad is None else f"LỖI: {bad}")
print(
    "Dung lượng:",
    round(os.path.getsize(backup_path) / 1024**2, 2),
    "MB"
)


In [ ]:

import os
import pandas as pd

BASE = "/kaggle/working/vsfc_llm"

search_dirs = [
    f"{BASE}/results/llm",
    f"{BASE}/results/remaining",
    "/kaggle/input/datasets/embehacker/vsfc-experiment-backup/results/llm",
    "/kaggle/input/datasets/embehacker/vsfc-experiment-backup/results/remaining"
]

print("=== KIỂM TRA FILE QWEN / PSEUDO LABEL ===")

found = []

for directory in search_dirs:
    if not os.path.isdir(directory):
        print("Chưa có thư mục:", directory)
        continue

    for root, dirs, files in os.walk(directory):
        for filename in files:
            name = filename.lower()

            if filename.endswith(".csv") and any(
                word in name
                for word in ["qwen", "pseudo", "pl_", "label"]
            ):
                path = os.path.join(root, filename)
                found.append(path)

                print("\nFILE:", path)

                try:
                    df = pd.read_csv(path, nrows=3)
                    print("Các cột:", df.columns.tolist())
                    print("Dữ liệu mẫu:")
                    display(df.head(2))
                except Exception as e:
                    print("Không đọc được:", str(e)[:150])

print("\nTỔNG SỐ FILE TÌM THẤY:", len(found))

if not found:
    print("CHƯA THẤY FILE QWEN / NHÃN GIẢ")


In [ ]:

import os
import torch
import pandas as pd
from datasets import load_from_disk

BASE = "/kaggle/working/vsfc_llm"
SOURCE = "/kaggle/input/datasets/embehacker/vsfc-experiment-backup/data"
PSEUDO_DIR = f"{BASE}/results/llm"

os.makedirs(PSEUDO_DIR, exist_ok=True)

# Chỉ đọc dữ liệu; không ghi cache vào Kaggle Input
DATA_PATH = f"{BASE}/data" if os.path.isfile(
    f"{BASE}/data/dataset_dict.json"
) else SOURCE

ds = load_from_disk(DATA_PATH)

assert torch.cuda.is_available(), "Chưa bật GPU"
assert len(ds["train"]) == 11426
assert len(ds["validation"]) == 1583

print("GPU:", torch.cuda.get_device_name(0))
print("Train:", len(ds["train"]))
print("Validation:", len(ds["validation"]))
print("Thư mục lưu:", PSEUDO_DIR)
print("SẴN SÀNG E5!")


In [ ]:

import torch
import numpy as np
import json
import os

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

QID = "Qwen/Qwen2.5-7B-Instruct"

qtok = AutoTokenizer.from_pretrained(QID)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

qmodel = AutoModelForCausalLM.from_pretrained(
    QID,
    quantization_config=quant_config,
    device_map="auto",
    dtype=torch.float16
)

qmodel.eval()

token_ids = [
    qtok.encode(c, add_special_tokens=False)
    for c in "ABC"
]
assert all(len(ids) == 1 for ids in token_ids)
token_ids = [ids[0] for ids in token_ids]

def prompt_p2(sentence):
    return (
        "Bạn là hệ thống phân loại cảm xúc phản hồi sinh viên bằng tiếng Việt.\n"
        "A: Tiêu cực — phàn nàn, không hài lòng, đánh giá không tốt.\n"
        "B: Trung tính — không thể hiện rõ khen hay chê.\n"
        "C: Tích cực — khen ngợi, hài lòng, đánh giá tốt.\n\n"
        f"Phản hồi:\n{sentence}\n\n"
        "Chỉ trả lời đúng một ký tự trong A, B, C."
    )

@torch.inference_mode()
def score_p2(sentence):
    messages = [
        {"role": "user", "content": prompt_p2(sentence)}
    ]

    formatted = qtok.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = qtok(
        formatted,
        return_tensors="pt",
        add_special_tokens=False
    ).to(qmodel.device)

    outputs = qmodel(**inputs)

    probabilities = torch.softmax(
        outputs.logits[0, -1, token_ids].float(),
        dim=-1
    )

    result = probabilities.cpu().numpy()

    del inputs, outputs
    return result

prior = np.mean(
    [score_p2(x) for x in ["N/A", "", "..."]],
    axis=0
)
prior = prior / prior.sum()

PSEUDO_DIR = "/kaggle/working/vsfc_llm/results/llm"
os.makedirs(PSEUDO_DIR, exist_ok=True)

with open(
    f"{PSEUDO_DIR}/qwen25_7b_pseudo_sentiment_prior_P2.json",
    "w"
) as f:
    json.dump({
        "prior": prior.tolist(),
        "model": QID,
        "prompt": "P2",
        "variant": "calibrated"
    }, f, indent=2)

print("\nĐÃ TẢI QWEN THÀNH CÔNG!")
print("Calibration prior:", prior)


In [ ]:

%pip install -q --upgrade "bitsandbytes>=0.46.1" accelerate

import importlib.metadata
import torch

print("bitsandbytes:", importlib.metadata.version("bitsandbytes"))
print("accelerate:", importlib.metadata.version("accelerate"))
print("GPU:", torch.cuda.is_available())


In [ ]:

import torch
import bitsandbytes as bnb

print("bitsandbytes:", bnb.__version__)
print("GPU:", torch.cuda.is_available())
print("Số GPU:", torch.cuda.device_count())
print("CÀI ĐẶT THÀNH CÔNG!")


In [ ]:

import torch
import numpy as np
import json
import os

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

QID = "Qwen/Qwen2.5-7B-Instruct"

qtok = AutoTokenizer.from_pretrained(QID)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

qmodel = AutoModelForCausalLM.from_pretrained(
    QID,
    quantization_config=quant_config,
    device_map="auto",
    dtype=torch.float16
)

qmodel.eval()

token_ids = [
    qtok.encode(c, add_special_tokens=False)
    for c in "ABC"
]
assert all(len(ids) == 1 for ids in token_ids)
token_ids = [ids[0] for ids in token_ids]

def prompt_p2(sentence):
    return (
        "Bạn là hệ thống phân loại cảm xúc phản hồi sinh viên bằng tiếng Việt.\n"
        "A: Tiêu cực — phàn nàn, không hài lòng, đánh giá không tốt.\n"
        "B: Trung tính — không thể hiện rõ khen hay chê.\n"
        "C: Tích cực — khen ngợi, hài lòng, đánh giá tốt.\n\n"
        f"Phản hồi:\n{sentence}\n\n"
        "Chỉ trả lời đúng một ký tự trong A, B, C."
    )

@torch.inference_mode()
def score_p2(sentence):
    messages = [
        {"role": "user", "content": prompt_p2(sentence)}
    ]

    formatted = qtok.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = qtok(
        formatted,
        return_tensors="pt",
        add_special_tokens=False
    ).to(qmodel.device)

    outputs = qmodel(**inputs)

    probabilities = torch.softmax(
        outputs.logits[0, -1, token_ids].float(),
        dim=-1
    )

    result = probabilities.cpu().numpy()

    del inputs, outputs
    return result

prior = np.mean(
    [score_p2(x) for x in ["N/A", "", "..."]],
    axis=0
)
prior = prior / prior.sum()

PSEUDO_DIR = "/kaggle/working/vsfc_llm/results/llm"
os.makedirs(PSEUDO_DIR, exist_ok=True)

with open(
    f"{PSEUDO_DIR}/qwen25_7b_pseudo_sentiment_prior_P2.json",
    "w"
) as f:
    json.dump({
        "prior": prior.tolist(),
        "model": QID,
        "prompt": "P2",
        "variant": "calibrated"
    }, f, indent=2)

print("\nĐÃ TẢI QWEN THÀNH CÔNG!")
print("Calibration prior:", prior)


In [1]:

import torch
import bitsandbytes as bnb
import transformers
from transformers.utils import is_bitsandbytes_available

print("GPU:", torch.cuda.is_available())
print("bitsandbytes:", bnb.__version__)
print("Transformers:", transformers.__version__)
print("Transformers nhận bitsandbytes:",
      is_bitsandbytes_available())


GPU: True
bitsandbytes: 0.50.2
Transformers: 5.16.1
Transformers nhận bitsandbytes: True


In [2]:

import torch
import bitsandbytes as bnb
import transformers

from transformers.utils import is_bitsandbytes_available

print("GPU:", torch.cuda.is_available())
print("bitsandbytes:", bnb.__version__)
print("Transformers:", transformers.__version__)
print(
    "Transformers nhận bitsandbytes:",
    is_bitsandbytes_available()
)


GPU: True
bitsandbytes: 0.50.2
Transformers: 5.16.1
Transformers nhận bitsandbytes: True


In [3]:

import os
import glob
import torch
from datasets import load_from_disk

BASE = "/kaggle/working/vsfc_llm"
PSEUDO_DIR = f"{BASE}/results/llm"

os.makedirs(PSEUDO_DIR, exist_ok=True)

# Tìm dữ liệu đã lưu trong Kaggle Input
candidates = [
    f"{BASE}/data",
    "/kaggle/input/datasets/embehacker/vsfc-experiment-backup/data"
]

candidates += glob.glob(
    "/kaggle/input/**/data/dataset_dict.json",
    recursive=True
)

DATA_PATH = None

for candidate in candidates:
    path = (
        os.path.dirname(candidate)
        if candidate.endswith("dataset_dict.json")
        else candidate
    )
    if os.path.isfile(f"{path}/dataset_dict.json"):
        DATA_PATH = path
        break

assert DATA_PATH is not None, (
    "Không tìm thấy dữ liệu UIT-VSFC. "
    "Kiểm tra Kaggle Add Input."
)

ds = load_from_disk(DATA_PATH)

assert len(ds["train"]) == 11426
assert len(ds["validation"]) == 1583
assert torch.cuda.is_available()

print("KHÔI PHỤC DỮ LIỆU THÀNH CÔNG!")
print("Đường dẫn:", DATA_PATH)
print("Train:", len(ds["train"]))
print("Validation:", len(ds["validation"]))
print("GPU:", torch.cuda.get_device_name(0))


KHÔI PHỤC DỮ LIỆU THÀNH CÔNG!
Đường dẫn: /kaggle/input/datasets/embehacker/vsfc-experiment-backup/data
Train: 11426
Validation: 1583
GPU: Tesla T4


In [4]:

import os
import gc
import json
import torch
import numpy as np

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

QID = "Qwen/Qwen2.5-7B-Instruct"

qtok = AutoTokenizer.from_pretrained(QID)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

qmodel = AutoModelForCausalLM.from_pretrained(
    QID,
    quantization_config=quant_config,
    device_map="auto",
    dtype=torch.float16
)

qmodel.eval()

token_ids = [
    qtok.encode(c, add_special_tokens=False)
    for c in "ABC"
]

assert all(len(x) == 1 for x in token_ids), (
    "Token A/B/C không còn là một token."
)

token_ids = [x[0] for x in token_ids]

def prompt_p2(sentence):
    return (
        "Bạn là hệ thống phân loại cảm xúc phản hồi sinh viên bằng tiếng Việt.\n"
        "A: Tiêu cực — phàn nàn, không hài lòng, đánh giá không tốt.\n"
        "B: Trung tính — không thể hiện rõ khen hay chê.\n"
        "C: Tích cực — khen ngợi, hài lòng, đánh giá tốt.\n\n"
        f"Phản hồi:\n{sentence}\n\n"
        "Chỉ trả lời đúng một ký tự trong A, B, C."
    )

@torch.inference_mode()
def score_p2(sentence):
    messages = [
        {"role": "user", "content": prompt_p2(sentence)}
    ]

    formatted = qtok.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = qtok(
        formatted,
        return_tensors="pt",
        add_special_tokens=False
    ).to(qmodel.device)

    output = qmodel(**inputs)

    probs = torch.softmax(
        output.logits[0, -1, token_ids].float(),
        dim=-1
    )

    result = probs.cpu().numpy()

    del inputs, output
    return result

# Calibration cố định, không dùng nhãn thật
prior = np.mean(
    [score_p2(s) for s in ["N/A", "", "..."]],
    axis=0
)

prior = prior / prior.sum()

with open(
    f"{PSEUDO_DIR}/qwen25_7b_pseudo_sentiment_prior_P2.json",
    "w"
) as f:
    json.dump({
        "model": QID,
        "prompt": "P2",
        "variant": "calibrated",
        "prior": prior.tolist()
    }, f, indent=2)

print("\nQWEN ĐÃ TẢI THÀNH CÔNG!")
print("Calibration prior:", prior)
print("GPU memory:")
for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}:",
        round(
            torch.cuda.memory_allocated(i) / 1024**3,
            2
        ),
        "GB"
    )


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]


QWEN ĐÃ TẢI THÀNH CÔNG!
Calibration prior: [0.8348967  0.1631842  0.00191918]
GPU memory:
GPU 0: 1.36 GB
GPU 1: 3.83 GB


In [5]:

import numpy as np

for i in range(10):
    sentence = ds["train"][i]["sentence"]

    probs = score_p2(sentence)

    calibrated = probs / (prior + 1e-12)
    calibrated = calibrated / calibrated.sum()

    print(
        f"Câu {i}: "
        f"raw={int(np.argmax(probs))} | "
        f"calibrated={int(np.argmax(calibrated))} | "
        f"confidence={float(np.max(calibrated)):.4f}"
    )

print("\nCHẠY THỬ 10 CÂU THÀNH CÔNG!")


Câu 0: raw=2 | calibrated=2 | confidence=0.9970
Câu 1: raw=2 | calibrated=2 | confidence=1.0000
Câu 2: raw=2 | calibrated=2 | confidence=0.9996
Câu 3: raw=0 | calibrated=0 | confidence=1.0000
Câu 4: raw=2 | calibrated=2 | confidence=1.0000
Câu 5: raw=2 | calibrated=2 | confidence=1.0000
Câu 6: raw=1 | calibrated=1 | confidence=0.9064
Câu 7: raw=0 | calibrated=0 | confidence=1.0000
Câu 8: raw=0 | calibrated=0 | confidence=1.0000
Câu 9: raw=0 | calibrated=0 | confidence=0.5042

CHẠY THỬ 10 CÂU THÀNH CÔNG!


In [6]:

import os
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

PSEUDO_DIR = "/kaggle/working/vsfc_llm/results/llm"
os.makedirs(PSEUDO_DIR, exist_ok=True)

def run_pseudo_batch(split, batch_size=250):
    assert split in ["train", "validation"]
    assert callable(globals().get("score_p2"))
    assert "prior" in globals()

    path = (
        f"{PSEUDO_DIR}/"
        f"qwen25_7b_pseudo_sentiment_{split}_P2.csv"
    )

    columns = [
        "sample_id", "sentence",
        "pseudo_raw", "pseudo_calibrated",
        "confidence", "p0", "p1", "p2"
    ]

    if os.path.isfile(path):
        old = pd.read_csv(path)
        assert set(columns).issubset(old.columns)
        old = old[columns].drop_duplicates(
            "sample_id", keep="last"
        )
        records = old.to_dict("records")
    else:
        records = []

    completed = {int(r["sample_id"]) for r in records}
    total = len(ds[split])

    # Chỉ xử lý batch_size câu chưa làm
    todo = [
        i for i in range(total)
        if i not in completed
    ][:batch_size]

    if not todo:
        print(f"{split}: ĐÃ HOÀN THÀNH {total}/{total}")
        return

    print(
        f"\nGÁN NHÃN {split.upper()}: "
        f"đã có {len(completed)}/{total}, "
        f"chạy thêm {len(todo)} câu"
    )

    for count, i in enumerate(tqdm(todo), start=1):
        sentence = ds[split][i]["sentence"]

        probs = score_p2(sentence)
        calibrated = probs / (prior + 1e-12)
        calibrated = calibrated / calibrated.sum()

        records.append({
            "sample_id": i,
            "sentence": sentence,
            "pseudo_raw": int(np.argmax(probs)),
            "pseudo_calibrated": int(np.argmax(calibrated)),
            "confidence": float(np.max(calibrated)),
            "p0": float(calibrated[0]),
            "p1": float(calibrated[1]),
            "p2": float(calibrated[2])
        })

        # Lưu an toàn mỗi 25 câu
        if count % 25 == 0 or count == len(todo):
            df = pd.DataFrame(records, columns=columns)
            df = df.sort_values("sample_id")
            tmp = path + ".tmp"
            df.to_csv(tmp, index=False)
            os.replace(tmp, path)

    done = len(pd.read_csv(path))

    print(f"\nĐÃ LƯU {split}: {done}/{total}")
    print("File:", path)

print("ĐÃ TẠO HÀM run_pseudo_batch!")


ĐÃ TẠO HÀM run_pseudo_batch!


In [7]:

run_pseudo_batch("train", batch_size=250)



GÁN NHÃN TRAIN: đã có 0/11426, chạy thêm 250 câu


  0%|          | 0/250 [00:00<?, ?it/s]


ĐÃ LƯU train: 250/11426
File: /kaggle/working/vsfc_llm/results/llm/qwen25_7b_pseudo_sentiment_train_P2.csv


In [8]:

import os
import pandas as pd
import zipfile

for split in ["train", "validation"]:
    path = (
        f"{PSEUDO_DIR}/"
        f"qwen25_7b_pseudo_sentiment_{split}_P2.csv"
    )

    n = len(pd.read_csv(path)) if os.path.isfile(path) else 0
    total = len(ds[split])

    print(f"{split}: {n}/{total}")

zip_path = "/kaggle/working/E5_pseudo_backup.zip"

with zipfile.ZipFile(
    zip_path, "w", zipfile.ZIP_DEFLATED
) as z:
    for filename in os.listdir(PSEUDO_DIR):
        if filename.startswith("qwen25_7b_pseudo_sentiment_"):
            z.write(
                os.path.join(PSEUDO_DIR, filename),
                arcname=filename
            )

print("\nĐÃ TẠO BACKUP:", zip_path)


train: 250/11426
validation: 0/1583

ĐÃ TẠO BACKUP: /kaggle/working/E5_pseudo_backup.zip


In [9]:

for batch in range(4):
    print(f"\n===== ĐỢT TRAIN {batch + 1}/4 =====")
    run_pseudo_batch("train", batch_size=250)

print("\nĐÃ HOÀN THÀNH THÊM TỐI ĐA 1.000 CÂU TRAIN!")



===== ĐỢT TRAIN 1/4 =====

GÁN NHÃN TRAIN: đã có 250/11426, chạy thêm 250 câu


  0%|          | 0/250 [00:00<?, ?it/s]


ĐÃ LƯU train: 500/11426
File: /kaggle/working/vsfc_llm/results/llm/qwen25_7b_pseudo_sentiment_train_P2.csv

===== ĐỢT TRAIN 2/4 =====

GÁN NHÃN TRAIN: đã có 500/11426, chạy thêm 250 câu


  0%|          | 0/250 [00:00<?, ?it/s]


ĐÃ LƯU train: 750/11426
File: /kaggle/working/vsfc_llm/results/llm/qwen25_7b_pseudo_sentiment_train_P2.csv

===== ĐỢT TRAIN 3/4 =====

GÁN NHÃN TRAIN: đã có 750/11426, chạy thêm 250 câu


  0%|          | 0/250 [00:00<?, ?it/s]


ĐÃ LƯU train: 1000/11426
File: /kaggle/working/vsfc_llm/results/llm/qwen25_7b_pseudo_sentiment_train_P2.csv

===== ĐỢT TRAIN 4/4 =====

GÁN NHÃN TRAIN: đã có 1000/11426, chạy thêm 250 câu


  0%|          | 0/250 [00:00<?, ?it/s]


ĐÃ LƯU train: 1250/11426
File: /kaggle/working/vsfc_llm/results/llm/qwen25_7b_pseudo_sentiment_train_P2.csv

ĐÃ HOÀN THÀNH THÊM TỐI ĐA 1.000 CÂU TRAIN!


In [10]:

import os
import pandas as pd
import zipfile

PSEUDO_DIR = "/kaggle/working/vsfc_llm/results/llm"

for split in ["train", "validation"]:
    path = (
        f"{PSEUDO_DIR}/"
        f"qwen25_7b_pseudo_sentiment_{split}_P2.csv"
    )

    n = len(pd.read_csv(path)) if os.path.isfile(path) else 0
    total = len(ds[split])

    print(f"{split}: {n}/{total}")

backup_path = "/kaggle/working/E5_pseudo_backup.zip"

with zipfile.ZipFile(
    backup_path, "w", zipfile.ZIP_DEFLATED
) as z:
    for filename in os.listdir(PSEUDO_DIR):
        if filename.startswith("qwen25_7b_pseudo_sentiment_"):
            z.write(
                os.path.join(PSEUDO_DIR, filename),
                arcname=filename
            )

with zipfile.ZipFile(backup_path) as z:
    assert z.testzip() is None

print("BACKUP OK:", backup_path)


train: 1250/11426
validation: 0/1583
BACKUP OK: /kaggle/working/E5_pseudo_backup.zip


In [11]:

for batch in range(4):
    print(f"\n===== TRAIN BATCH {batch + 1}/4 =====")
    run_pseudo_batch("train", batch_size=250)

print("\nHOÀN THÀNH NHÓM TRAIN!")



===== TRAIN BATCH 1/4 =====

GÁN NHÃN TRAIN: đã có 1250/11426, chạy thêm 250 câu


  0%|          | 0/250 [00:00<?, ?it/s]


ĐÃ LƯU train: 1500/11426
File: /kaggle/working/vsfc_llm/results/llm/qwen25_7b_pseudo_sentiment_train_P2.csv

===== TRAIN BATCH 2/4 =====

GÁN NHÃN TRAIN: đã có 1500/11426, chạy thêm 250 câu


  0%|          | 0/250 [00:00<?, ?it/s]


ĐÃ LƯU train: 1750/11426
File: /kaggle/working/vsfc_llm/results/llm/qwen25_7b_pseudo_sentiment_train_P2.csv

===== TRAIN BATCH 3/4 =====

GÁN NHÃN TRAIN: đã có 1750/11426, chạy thêm 250 câu


  0%|          | 0/250 [00:00<?, ?it/s]


ĐÃ LƯU train: 2000/11426
File: /kaggle/working/vsfc_llm/results/llm/qwen25_7b_pseudo_sentiment_train_P2.csv

===== TRAIN BATCH 4/4 =====

GÁN NHÃN TRAIN: đã có 2000/11426, chạy thêm 250 câu


  0%|          | 0/250 [00:00<?, ?it/s]


ĐÃ LƯU train: 2250/11426
File: /kaggle/working/vsfc_llm/results/llm/qwen25_7b_pseudo_sentiment_train_P2.csv

HOÀN THÀNH NHÓM TRAIN!


In [14]:

import os
import zipfile
import pandas as pd

PSEUDO_DIR = "/kaggle/working/vsfc_llm/results/llm"
BACKUP = "/kaggle/working/E5_pseudo_backup.zip"

for split in ["train", "validation"]:
    path = f"{PSEUDO_DIR}/qwen25_7b_pseudo_sentiment_{split}_P2.csv"
    n = len(pd.read_csv(path)) if os.path.isfile(path) else 0
    print(f"{split}: {n}/{len(ds[split])}")

with zipfile.ZipFile(BACKUP, "w", zipfile.ZIP_DEFLATED) as z:
    for filename in os.listdir(PSEUDO_DIR):
        if filename.startswith("qwen25_7b_pseudo_sentiment_"):
            z.write(
                os.path.join(PSEUDO_DIR, filename),
                arcname=filename
            )

with zipfile.ZipFile(BACKUP) as z:
    assert z.testzip() is None

print("BACKUP OK:", BACKUP)


train: 3250/11426
validation: 0/1583
BACKUP OK: /kaggle/working/E5_pseudo_backup.zip


In [13]:

for batch in range(4):
    print(f"\n===== TRAIN BATCH {batch + 1}/4 =====")
    run_pseudo_batch("train", batch_size=250)

print("\nHOÀN THÀNH THÊM 1.000 CÂU!")



===== TRAIN BATCH 1/4 =====

GÁN NHÃN TRAIN: đã có 2250/11426, chạy thêm 250 câu


  0%|          | 0/250 [00:00<?, ?it/s]


ĐÃ LƯU train: 2500/11426
File: /kaggle/working/vsfc_llm/results/llm/qwen25_7b_pseudo_sentiment_train_P2.csv

===== TRAIN BATCH 2/4 =====

GÁN NHÃN TRAIN: đã có 2500/11426, chạy thêm 250 câu


  0%|          | 0/250 [00:00<?, ?it/s]


ĐÃ LƯU train: 2750/11426
File: /kaggle/working/vsfc_llm/results/llm/qwen25_7b_pseudo_sentiment_train_P2.csv

===== TRAIN BATCH 3/4 =====

GÁN NHÃN TRAIN: đã có 2750/11426, chạy thêm 250 câu


  0%|          | 0/250 [00:00<?, ?it/s]


ĐÃ LƯU train: 3000/11426
File: /kaggle/working/vsfc_llm/results/llm/qwen25_7b_pseudo_sentiment_train_P2.csv

===== TRAIN BATCH 4/4 =====

GÁN NHÃN TRAIN: đã có 3000/11426, chạy thêm 250 câu


  0%|          | 0/250 [00:00<?, ?it/s]


ĐÃ LƯU train: 3250/11426
File: /kaggle/working/vsfc_llm/results/llm/qwen25_7b_pseudo_sentiment_train_P2.csv

HOÀN THÀNH THÊM 1.000 CÂU!


In [15]:

import os
import pandas as pd

TRAIN_FILE = (
    "/kaggle/working/vsfc_llm/results/llm/"
    "qwen25_7b_pseudo_sentiment_train_P2.csv"
)

TOTAL = len(ds["train"])

while True:
    completed = (
        len(pd.read_csv(TRAIN_FILE))
        if os.path.isfile(TRAIN_FILE)
        else 0
    )

    if completed >= TOTAL:
        print("\nHOÀN THÀNH TRAIN:", completed, "/", TOTAL)
        break

    print(f"\nTiến độ: {completed}/{TOTAL}")
    run_pseudo_batch("train", batch_size=250)

print("\n✅ ĐÃ GÁN NHÃN XONG TOÀN BỘ TRAIN!")



Tiến độ: 3250/11426

GÁN NHÃN TRAIN: đã có 3250/11426, chạy thêm 250 câu


  0%|          | 0/250 [00:00<?, ?it/s]


ĐÃ LƯU train: 3500/11426
File: /kaggle/working/vsfc_llm/results/llm/qwen25_7b_pseudo_sentiment_train_P2.csv

Tiến độ: 3500/11426

GÁN NHÃN TRAIN: đã có 3500/11426, chạy thêm 250 câu


  0%|          | 0/250 [00:00<?, ?it/s]


ĐÃ LƯU train: 3750/11426
File: /kaggle/working/vsfc_llm/results/llm/qwen25_7b_pseudo_sentiment_train_P2.csv

Tiến độ: 3750/11426

GÁN NHÃN TRAIN: đã có 3750/11426, chạy thêm 250 câu


...


ĐÃ LƯU train: 11000/11426
File: /kaggle/working/vsfc_llm/results/llm/qwen25_7b_pseudo_sentiment_train_P2.csv

Tiến độ: 11000/11426

GÁN NHÃN TRAIN: đã có 11000/11426, chạy thêm 250 câu


  0%|          | 0/250 [00:00<?, ?it/s]


ĐÃ LƯU train: 11250/11426
File: /kaggle/working/vsfc_llm/results/llm/qwen25_7b_pseudo_sentiment_train_P2.csv

Tiến độ: 11250/11426

GÁN NHÃN TRAIN: đã có 11250/11426, chạy thêm 176 câu


  0%|          | 0/176 [00:00<?, ?it/s]


ĐÃ LƯU train: 11426/11426
File: /kaggle/working/vsfc_llm/results/llm/qwen25_7b_pseudo_sentiment_train_P2.csv

HOÀN THÀNH TRAIN: 11426 / 11426

✅ ĐÃ GÁN NHÃN XONG TOÀN BỘ TRAIN!


In [4]:

from pathlib import Path

p = Path(
    "/kaggle/working/vsfc_llm/results/llm/"
    "qwen25_7b_pseudo_sentiment_train_P2.csv"
)

print("File tồn tại:", p.exists())

if p.exists():
    import pandas as pd
    df = pd.read_csv(p)
    print("Số câu:", len(df))
    display(df.head())
else:
    print("File không còn ở đường dẫn này.")


File tồn tại: False
File không còn ở đường dẫn này.


In [6]:

from datasets import load_from_disk
import os

DATA_PATH = (
    "/kaggle/input/datasets/embehacker/"
    "vsfc-experiment-backup/data"
)

ds = load_from_disk(DATA_PATH)

print("KHÔI PHỤC DỮ LIỆU THÀNH CÔNG!")
print("Train:", len(ds["train"]))
print("Validation:", len(ds["validation"]))
print("Test:", len(ds["test"]))

assert len(ds["train"]) == 11426
assert len(ds["validation"]) == 1583
assert len(ds["test"]) == 3166


KHÔI PHỤC DỮ LIỆU THÀNH CÔNG!
Train: 11426
Validation: 1583
Test: 3166


In [7]:

import os

print("Dữ liệu gốc:", "ds" in globals())
print("Hàm gán nhãn:", callable(globals().get("run_pseudo_batch")))
print("Mô hình Qwen:", "qmodel" in globals())
print("Hàm dự đoán:", callable(globals().get("score_p2")))

TRAIN_FILE = (
    "/kaggle/working/vsfc_llm/results/llm/"
    "qwen25_7b_pseudo_sentiment_train_P2.csv"
)

print("CSV train E5:", os.path.isfile(TRAIN_FILE))


Dữ liệu gốc: True
Hàm gán nhãn: False
Mô hình Qwen: False
Hàm dự đoán: False
CSV train E5: False


In [8]:

import torch
import importlib.metadata as md

print("GPU:", torch.cuda.is_available())
print("Số GPU:", torch.cuda.device_count())

for lib in ["transformers", "bitsandbytes", "accelerate"]:
    try:
        print(lib, md.version(lib))
    except Exception:
        print(lib, "CHƯA CÀI")

print("Train:", len(ds["train"]))
print("Validation:", len(ds["validation"]))


GPU: True
Số GPU: 2
transformers 5.16.1
bitsandbytes CHƯA CÀI
accelerate 1.14.0
Train: 11426
Validation: 1583


In [ ]:

%pip install -q --no-cache-dir "bitsandbytes>=0.46.1"

import importlib.metadata as md

print("bitsandbytes:", md.version("bitsandbytes"))
print("transformers:", md.version("transformers"))
print("accelerate:", md.version("accelerate"))
print("CÀI ĐẶT XONG!")

